# 4 - FWI is just training a model (PyTorch nn.Module style)

Full-waveform inversion looks exotic, but it is the **same loop as neural-network training**. The only unusual part is that the 'network' is a wave-equation simulator instead of a stack of linear layers - everything else is standard PyTorch.

| Neural-network training | This FWI |
|---|---|
| model (`nn.Module`) | the differentiable wave-equation solver |
| weights (`nn.Parameter`) | the material model `alpha2` (squared wave speed) |
| forward pass | solve the wave equation -> predicted seismograms |
| loss | L2 waveform misfit vs the recorded data |
| `loss.backward()` | the adjoint-state gradient, computed automatically |
| `optimizer.step()` | update the model |
| training data | seismograms recorded through the true (cracked) plate |

We train **the same model on the same data twice**, changing only the optimizer: **Variant A** full-batch **L-BFGS** (second-order) and **Variant B** mini-batch **Adam** (first-order, stochastic) - so the loss curves are a genuine optimizer-vs-optimizer comparison.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/04_fwi_as_nn_training.ipynb)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    OWNER = "seidlr"  # change to your account if you forked
    !git clone -q https://github.com/{OWNER}/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    sys.path.insert(0, os.path.abspath("src"))  # make `import fwi` importable now

import torch
from torch import nn
import matplotlib.pyplot as plt
from fwi.config import resolve_device, resolve_dtype

device = resolve_device()
dtype = resolve_dtype(device)
print("device:", device, "| dtype:", dtype)

## The 'network': a differentiable wave solver

This is the forward model: it time-steps the 2D acoustic wave equation for all shots at once (one `(S, nI, nJ)` batch) and returns the receiver traces. It is pure PyTorch, so autograd differentiates the misfit straight through every timestep - that is the adjoint-state gradient, for free. Its source:

In [ ]:
import inspect
from fwi.forward import forward_multishot
print(inspect.getsource(forward_multishot))

## Wrap it as an `nn.Module`

The learnable weight is a dimensionless model `m = alpha2 / alpha2_background` (so it is ~1, like well-scaled NN weights). `forward()` takes a batch of shots (source indices), turns `m` back into a physical `alpha2`, and runs the solver. The `clamp` keeps every step CFL-stable and the `mask` freezes the fixed boundary cells - both differentiable, so the training loops stay plain PyTorch.

In [ ]:
from fwi.forward import forward_multishot

class MultiShotFWI(nn.Module):
    """The wave solver as a trainable model: weights = the material model.

    forward(src_i, src_j) runs the shots at those source positions and returns
    their seismograms (shape (len(src_i), n_receivers, nt)).
    """

    def __init__(self, m_init, src_sig, rec_i, rec_j, cfg, active_mask, m_max, alpha2_bg):
        super().__init__()
        self.m = nn.Parameter(m_init.clone())          # what we invert for
        self.register_buffer('src_sig', src_sig)
        self.register_buffer('rec_i', rec_i)
        self.register_buffer('rec_j', rec_j)
        self.register_buffer('mask', active_mask.to(m_init.dtype))
        self.cfg, self.m_max, self.alpha2_bg = cfg, m_max, alpha2_bg

    def forward(self, src_i, src_j):
        # dimensionless m -> CFL-safe, boundary-masked physical alpha2
        alpha2 = self.m.clamp(0.0, self.m_max) * self.mask * self.alpha2_bg
        return forward_multishot(alpha2, self.src_sig, src_i, src_j,
                                 self.rec_i, self.rec_j, self.cfg)

## Load the data and define the loss

`build_multishot_batched('crack')` gives a **moving-source** acquisition: 12 shots (each a source at one boundary sensor, recorded at the others) through the true cracked plate - our training data. The shots are the dataset. The loss is the L2 waveform misfit, normalised by its starting value `J0` so it begins at 1.0; `model_misfit(model, idx)` evaluates it over a subset of shots (a mini-batch) or all of them (full batch).

In [ ]:
from fwi.problems import build_multishot_batched
from fwi.config import SimConfig, SPEED_ALUMINUM

mp = build_multishot_batched('crack', device=device, dtype=dtype, n_shots=12)
n_shots = int(mp.src_i.shape[0])
alpha2_bg = SPEED_ALUMINUM ** 2
m_init = mp.start_alpha2 / alpha2_bg
c_limit = mp.cfg.cfl * min(mp.cfg.dx_m, mp.cfg.dy_m) / mp.cfg.dt
m_max = 0.9 * c_limit ** 2 / alpha2_bg

def model_misfit(model, idx=None):
    """0.5 * sum|pred - observed|^2 * dt over all shots (idx=None) or a batch."""
    si = mp.src_i if idx is None else mp.src_i[idx]
    sj = mp.src_j if idx is None else mp.src_j[idx]
    obs = mp.observed if idx is None else mp.observed[idx]
    sm = mp.self_mask if idx is None else mp.self_mask[idx]
    resid = (model(si, sj) - obs) * sm[..., None]
    return 0.5 * (resid ** 2).sum() * mp.cfg.dt

model_for_J0 = MultiShotFWI(m_init, mp.src_sig, mp.rec_i, mp.rec_j, mp.cfg,
                            mp.active_mask, m_max, alpha2_bg).to(device)
with torch.no_grad():
    J0 = float(model_misfit(model_for_J0))          # full-data initial misfit
print(f'dataset: {n_shots} shots | trainable params: {model_for_J0.m.numel()} | start J/J0 = 1.000')

## Variant A: full-batch L-BFGS (second-order)

Use **all 12 shots every step**. `torch.optim.LBFGS` takes a `closure` that re-evaluates the model and calls `loss.backward()` - the standard second-order PyTorch pattern. With many shots the summed-gradient line search needs a few inner iterations per step to stay well-conditioned (`max_iter=3`), otherwise it stalls; each outer step is one logged 'epoch'.

In [ ]:
modelA = MultiShotFWI(m_init, mp.src_sig, mp.rec_i, mp.rec_j, mp.cfg,
                      mp.active_mask, m_max, alpha2_bg).to(device)
optA = torch.optim.LBFGS(modelA.parameters(), lr=1.0, max_iter=3,
                         line_search_fn='strong_wolfe')
losses_lbfgs = []

def closure():
    optA.zero_grad()
    loss = model_misfit(modelA) / J0          # full batch: all 12 shots
    loss.backward()
    return loss

for epoch in range(10):
    optA.step(closure)
    with torch.no_grad():
        full = float(model_misfit(modelA)) / J0
    losses_lbfgs.append(full)
    print(f'epoch {epoch + 1:02d}/10 | full J/J0 = {full:.4e}')

print(f'\nL-BFGS reduced {losses_lbfgs[0] / losses_lbfgs[-1]:.1f}x')

## Variant B: mini-batch Adam (stochastic FWI)

Same model, same data - now the deep-learning way. The shots are the dataset, so we shuffle them, take **mini-batches**, and step **Adam** (first-order) on each batch's stochastic gradient. No closure: just `zero_grad -> forward -> loss -> backward -> step`, looped over batches and epochs.

| Deep-learning training | Stochastic FWI (Variant B) | Full-batch L-BFGS (Variant A) |
|---|---|---|
| dataset of examples | the shots (source positions) | the same shots |
| a mini-batch | a random subset of shots | all the shots, every step (full batch) |
| `Adam` on batch gradient | `Adam` on the batch's summed-misfit gradient | `L-BFGS` on the full-data gradient (second-order) |
| shuffle each epoch | re-shuffle the shots each epoch | no shuffle - deterministic |
| epoch validation loss | full-data `J/J0` evaluated each epoch | same full `J/J0`, logged each step |

In [ ]:
import random
random.seed(0)
modelB = MultiShotFWI(m_init, mp.src_sig, mp.rec_i, mp.rec_j, mp.cfg,
                      mp.active_mask, m_max, alpha2_bg).to(device)
optB = torch.optim.Adam(modelB.parameters(), lr=0.008)
BATCH = 4
ids = list(range(n_shots))
losses_adam = []

for epoch in range(25):
    random.shuffle(ids)                       # reshuffle the 'dataset' each epoch
    for k in range(0, n_shots, BATCH):        # iterate mini-batches
        b = torch.tensor(ids[k:k + BATCH], device=device)
        optB.zero_grad()
        loss = model_misfit(modelB, b) / J0   # one mini-batch of shots
        loss.backward()
        optB.step()
    with torch.no_grad():                     # epoch loss over ALL shots
        full = float(model_misfit(modelB)) / J0
    losses_adam.append(full)
    print(f'epoch {epoch + 1:02d}/25 | full J/J0 = {full:.4e}')

print(f'\nAdam reduced {losses_adam[0] / losses_adam[-1]:.1f}x')

## L-BFGS vs Adam - same data, just the optimizer

Both train `MultiShotFWI` on the identical 12-shot dataset, so this isolates the optimizer. L-BFGS (second-order, full-batch) curves down in a handful of epochs; Adam (first-order, mini-batch) takes many more but cheaper steps and shows the stochastic ripple - the classic trade-off from neural-network training. Both recover the crack. (Caveat: each L-BFGS epoch runs a few inner line-search iterations, so it does more forward solves per epoch than one Adam epoch.)

In [ ]:
with torch.no_grad():
    recA = ((modelA.m.clamp(0.0, m_max) * modelA.mask * alpha2_bg) - mp.start_alpha2).cpu().numpy()
    recB = ((modelB.m.clamp(0.0, m_max) * modelB.mask * alpha2_bg) - mp.start_alpha2).cpu().numpy()
truth = (mp.true_alpha2 - mp.start_alpha2).cpu().numpy()
vmax = abs(truth).max()

fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
ax[0].semilogy(range(1, len(losses_lbfgs) + 1), losses_lbfgs, 'o-', label='L-BFGS (full-batch)')
ax[0].semilogy(range(1, len(losses_adam) + 1), losses_adam, 's-', label='Adam (mini-batch)')
ax[0].set(title='same 12-shot data, two optimizers', xlabel='epoch', ylabel='full J / J0')
ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].imshow(recA, cmap='RdBu_r', vmin=-vmax, vmax=vmax); ax[1].set_title('L-BFGS recovered')
ax[2].imshow(recB, cmap='RdBu_r', vmin=-vmax, vmax=vmax); ax[2].set_title('Adam recovered')
ax[3].imshow(truth, cmap='RdBu_r', vmin=-vmax, vmax=vmax); ax[3].set_title('true crack')
for a in ax[1:]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()